# IndicTrans2 finetuning — one experiment settings cell

Edit the next cell; run top to bottom in a **fresh** Kaggle/Colab session. Default native Maithili→Hindi; the same corpus-conversion/splitting interface as NLLB. IndicTrans2 needs its official toolkit and optional custom code, so training is **OFF until you enable it**. Development loss/best-checkpoint selection is built into the trainer. Reporting and test-set inference are optional.

See `docs/experiment-guide.md` and `docs/notebook-platforms.md`.


## 1. Experiment settings — edit only here


In [ ]:
# EDIT ONLY THIS CELL FOR AN INDIC TRANS2 EXPERIMENT.
EXPERIMENT = {
    "data_mode": "aligned_text",   # "aligned_text", "tsv", "jsonl", "pre_split", "example".
    "source_lang": "mai_Deva",     # Native IndicTrans2 (Maithili).
    "target_lang": "hin_Deva",     # Native IndicTrans2 (Hindi).
    "source_text": "/kaggle/input/YOUR_DATASET/mai.txt",
    "target_text": "/kaggle/input/YOUR_DATASET/hin.txt",
    "tsv_file": "/kaggle/input/YOUR_DATASET/parallel.tsv",
    "tsv_has_header": False,
    "parallel_jsonl": "data/local/mai-hin/parallel.jsonl",
    "split_directory": "data/local/mai-hin/splits",
    "dataset_id": "corpus-v1",
    "split_seed": 42, "split_ratios": (0.80, 0.10, 0.10),
    "group_by_document": True,
    "validate_data": False,       # Full audit; basic validity checks always run.
    "data_report": False,         # Extra corpus plots for a paper.
    "model_id": None,             # None = official IT2 distilled checkpoint for direction.
    "conditioning": "native",     # "text_prefix"/"dedicated_token" are experimental.
    "proxy_source_lang": None, "proxy_target_lang": None,  # For unsupported language tags.
    "finetune_mode": "lora",       # Dedicated tokens require "full".
    "train_seed": 42,
    "epochs": 3, "max_steps": -1, # max_steps=1 is a smoke test.
    "batch_size": 1, "gradient_accumulation_steps": 16,
    "learning_rate": 2e-4, "max_length": 128,
    "max_new_tokens": 128, "beams": 4,
    "precision": "fp16", "gradient_checkpointing": True, "lora_rank": 8,
    "run_label": "pilot-1",
    "allow_training": False,      # Set True after installing official IndicTransToolkit.
    "allow_remote_code": False,   # Must explicitly review/trust upstream HF custom code.
    "pretrain_baseline": False,   # Optional costly untouched-model dev generation.
    "baseline_dev_size": 100,
    "posttrain_test": False, "publication_report": False,
    "test_baseline": False,
    "save_archive": True, "include_checkpoints": False,
    "setup_mode": "train",        # GPU / pinned Transformers+PEFT stack.
    "refresh_repo": False,
}


## 2. Environment

Choose GPU first. Install reviewed IndicTransToolkit separately using the guide.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
ROOT = (Path("/kaggle/working/BhaashikMTFinetuning")
        if Path("/kaggle/working").is_dir()
        else Path("/content/BhaashikMTFinetuning")
        if Path("/content").is_dir() else Path.cwd().resolve())
os.environ["USE_TF"], os.environ["USE_FLAX"] = "0", "0"
if not (ROOT / "pyproject.toml").is_file():
    if ROOT.exists(): raise ValueError(f"Repository checkout is incomplete: {ROOT}")
    subprocess.run(["git", "clone", "https://github.com/bhaashik/BhaashikMTFinetuning.git",
                    str(ROOT)], check=True)
elif EXPERIMENT["refresh_repo"]:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
os.chdir(ROOT)
if any(m in sys.modules for m in ("torch","transformers","peft","datasets")):
    raise RuntimeError("Restart the notebook kernel before installing model packages.")
cmd = [sys.executable, str(ROOT / "scripts" / "setup_notebook.py"),
       "--extras", EXPERIMENT["setup_mode"]]
if EXPERIMENT["setup_mode"].startswith("train"):
    cmd.append("--require-cuda")
subprocess.run(cmd, check=True)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from bhaashik.experiment_workflow import check_settings
check_settings(EXPERIMENT, model_family="indictrans2")
print("Repo:", ROOT)
# IMPORTANT: IndicTransToolkit is an ADDITIONAL dependency. Install its
# official reviewed version in this fresh session before model inference/training.
# See docs/experiment-guide.md; avoid mixing incompatible model stacks.


## 3. Prepare parallel corpus

Convert aligned text/TSV/JSONL, group and split, record hashes.


In [ ]:
from bhaashik.experiment_workflow import (
    prepare_parallel_data, experiment_paths, training_parameters,
    write_experiment_config, report_data_if_enabled,
)
PREPARED = prepare_parallel_data(ROOT, EXPERIMENT, model_family="indictrans2")
PATHS = experiment_paths(ROOT, EXPERIMENT, family="indictrans2", prepared=PREPARED)
DATA, RUN, REPORT_DIR = PREPARED["data_dir"], PATHS["run"], PATHS["report"]
report_data_if_enabled(EXPERIMENT, PREPARED, REPORT_DIR)
TRAIN_CONFIG = training_parameters(EXPERIMENT, PREPARED, PATHS,
                                   model_family="indictrans2")
TRAIN_CONFIG_PATH = write_experiment_config(EXPERIMENT, PREPARED, PATHS, TRAIN_CONFIG)
print("Model:", EXPERIMENT["model_id"] or "IT2 official distilled checkpoint")
print("Output:", RUN)


## 4. Optional untouched-model development baseline

Off by default; not required for per-epoch development loss.


In [ ]:
if EXPERIMENT["pretrain_baseline"]:
    from bhaashik.model_comparisons import select_dev_sample, run_indictrans2_dev
    from bhaashik.indictrans2_adaptation import processing_pair
    assert EXPERIMENT["conditioning"] == "native", "Original baseline requires native IT2 tags."
    processing_pair(EXPERIMENT["source_lang"], EXPERIMENT["target_lang"], strategy="native")
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    sample = REPORT_DIR / "dev_sample.jsonl"
    if not sample.exists():
        select_dev_sample(DATA / "dev.jsonl", sample,
                          sample_size=EXPERIMENT["baseline_dev_size"],
                          seed=EXPERIMENT["split_seed"])
    prediction = REPORT_DIR / "indictrans2-original-dev.jsonl"
    if not prediction.exists():
        run_indictrans2_dev(
            sample, prediction, EXPERIMENT["source_lang"], EXPERIMENT["target_lang"],
            allow_remote_code=EXPERIMENT["allow_remote_code"])
    from bhaashik.pretrain_review import compare_dev
    print(compare_dev(sample, {"IndicTrans2 original": prediction}, REPORT_DIR))
else:
    print("Pre-training dev inference OFF (not needed for training loss).")


## 5. Train

Off until you enable `allow_training` and explicitly allow trusted custom model code.


In [ ]:
if EXPERIMENT["allow_training"]:
    assert EXPERIMENT["setup_mode"].startswith("train")
    assert EXPERIMENT["allow_remote_code"], (
        "Review official IndicTrans2 custom model code; set allow_remote_code=True.")
    from bhaashik.indictrans2_adaptation import train_indictrans2
    # Evaluation loss is computed each epoch; the lowest-loss checkpoint is restored.
    print("Development loss:", train_indictrans2(TRAIN_CONFIG_PATH))
    print("Saved model:", RUN)
else:
    print("IT2 training OFF until IndicTransToolkit/custom-code access is prepared.")


## 6. Optional final test and publication metrics

Off by default. The test split is not used during model selection.


In [ ]:
if EXPERIMENT["posttrain_test"]:
    assert (RUN / "run_manifest.json").exists(), "Train or attach a model first."
    from bhaashik.indictrans2_adaptation import generate_indictrans2
    preds = RUN / "predictions.jsonl"
    if not preds.exists():
        generate_indictrans2(
            RUN, DATA / "test.jsonl", preds,
            strategy=EXPERIMENT["conditioning"],
            proxy_src=EXPERIMENT["proxy_source_lang"],
            proxy_tgt=EXPERIMENT["proxy_target_lang"],
            batch_size=EXPERIMENT["batch_size"],
            max_length=EXPERIMENT["max_length"],
            max_new_tokens=EXPERIMENT["max_new_tokens"],
            beams=EXPERIMENT["beams"],
            allow_remote_code=EXPERIMENT["allow_remote_code"],
        )
    print("Final held-out predictions:", preds)
    if EXPERIMENT["publication_report"]:
        from bhaashik.evaluation import evaluate, export_errors
        print(evaluate(preds, RUN / "evaluation.json")["scores"])
        export_errors(preds, RUN / "annotations.csv", seed=EXPERIMENT["train_seed"])
else:
    print("Final test generation and publication metrics OFF.")


## 7. Preserve outputs

Fast model ZIP; include Trainer checkpoints only for resumability.


In [ ]:
from bhaashik.experiment_workflow import save_optional_archive
if (RUN / "run_manifest.json").exists():
    archive = save_optional_archive(ROOT, EXPERIMENT, PATHS,
                                    report=REPORT_DIR if REPORT_DIR.is_dir() else None)
    if archive: print("Preserve archive:", archive)
else:
    print("No trained model yet; archive step skipped.")
